In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

REPO_ROOT   = Path("..").resolve()
SCRIPTS_DIR = REPO_ROOT / "scripts"
RESULTS_DIR = REPO_ROOT / "results"
ADBENCH_DIR = REPO_ROOT / "data" / "adbench"

# Table 9 results go in their own subfolder to avoid
# colliding with the Appendix A benchmark results
ABLATION_RESULTS_DIR = RESULTS_DIR / "ablation_adbench"

sys.path.insert(0, str(SCRIPTS_DIR))

print("Repo root            :", REPO_ROOT)
print("ADBench dir          :", ADBENCH_DIR, "| exists:", ADBENCH_DIR.exists())
print("Ablation results dir :", ABLATION_RESULTS_DIR)

if ADBENCH_DIR.exists():
    npz_files = list(ADBENCH_DIR.glob("*.npz"))
    print(f"ADBench datasets found: {len(npz_files)}")


Repo root            : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB
ADBench dir          : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\data\adbench | exists: True
Ablation results dir : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\results\ablation_adbench
ADBench datasets found: 28


In [2]:
# Set RUN_TRAINING = True  to re-run the 3-config ADBench replication
# Set RUN_TRAINING = False to load pre-computed results (default)
#
# WARNING: RUN_TRAINING = True requires an NVIDIA GPU and approximately
# 6 hours of compute time (3 configurations x 3 seeds x 28 datasets).

RUN_TRAINING = True

# Seeds used in the paper — do not change if you want to reproduce Table 9
SEEDS = [69, 72, 888]


# ============================================================
# CELL 3 — Optional: run ADBench replication from scratch
# ============================================================

if RUN_TRAINING:
    if not ADBENCH_DIR.exists() or not list(ADBENCH_DIR.glob("*.npz")):
        raise FileNotFoundError(
            f"\nADBench datasets not found at:\n  {ADBENCH_DIR}\n\n"
            "Please download from:\n"
            "  https://github.com/Minqi824/ADBench\n"
            "and place all .npz files in data/adbench/"
        )

    ABLATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    from caldm_ablation_adbench import run_adbench_ablation

    print("Running ADBench ablation replication...")
    print(f"Seeds  : {SEEDS}")
    print(f"Output : {ABLATION_RESULTS_DIR}")
    print("Estimated time: approximately 6 hours on an NVIDIA RTX-class GPU.\n")

    run_adbench_ablation(
        input_dir=str(ADBENCH_DIR),
        output_dir=str(ABLATION_RESULTS_DIR),
        seeds=SEEDS,
        verbose=True,
    )

    print("\nReplication complete. Results saved to:", ABLATION_RESULTS_DIR)

else:
    print("RUN_TRAINING = False — loading pre-computed results.")


# ============================================================
# CELL 4 — Load results
# ============================================================

summary_path    = ABLATION_RESULTS_DIR / "adbench_summary.csv"
per_run_path    = ABLATION_RESULTS_DIR / "adbench_per_run.csv"
per_dataset_path = ABLATION_RESULTS_DIR / "adbench_per_dataset.csv"

if not summary_path.exists():
    raise FileNotFoundError(
        f"Results not found at {summary_path}.\n"
        "Either set RUN_TRAINING = True in Cell 2, or ensure the pre-computed\n"
        "results CSVs are present in results/ablation_adbench/"
    )

summary     = pd.read_csv(summary_path)
per_run     = pd.read_csv(per_run_path)
per_dataset = pd.read_csv(per_dataset_path)

print(f"Loaded summary     : {len(summary)} rows  (expect 3 — configs C, E, F)")
print(f"Loaded per_run     : {len(per_run)} rows")
print(f"Loaded per_dataset : {len(per_dataset)} rows  (expect 28 — one per dataset)")
print()
print("Configs found:")
for _, r in summary.iterrows():
    print(f"  {r['config']:<30}  {r['label']}")


Running ADBench ablation replication...
Seeds  : [69, 72, 888]
Output : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\results\ablation_adbench
Estimated time: approximately 6 hours on an NVIDIA RTX-class GPU.

[setup] device=cuda  seeds=[69, 72, 888]  data_fraction=1.0
[setup] found 28 .npz files in C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\data\adbench

[1/28] 10_cover  (286048 rows, 10 features, 0.96% outliers)
  hp: hidden=64 latent=4 context=4 heads=4 batch=256
  Latent Diff (Uncond.) seed=69: AUC=0.780 AP=0.023 (964.8s)
  Latent Diff (Uncond.) seed=72: AUC=0.827 AP=0.031 (878.3s)
  Latent Diff (Uncond.) seed=888: AUC=0.816 AP=0.031 (859.4s)
  CALDM-D (proposed) seed=69: AUC=0.932 AP=0.070 (2455.0s)
  CALDM-D (proposed) seed=72: AUC=0.908 AP=0.053 (2398.2s)
  CALDM-D (proposed) seed=888: AUC=0.931 AP=0.069 (2378.5s)
  CALDM-J (jointly cond.) seed=69:

In [3]:
LABEL_MAP = {
    "C_vae_diff_nocxt": "C. Latent Diff (Uncond.)",
    "E_cxt_in_diff":    "E. CALDM-D (proposed)",
    "F_full_caldm":     "F. CALDM-J",
}

PROPOSED = "E_cxt_in_diff"

def fmt(mean, std, d=3):
    return f"{mean:.{d}f} \u00b1 {std:.{d}f}"

def fmt_pval(val):
    if pd.isna(val): return "\u2014"
    if val < 0.001:  return f"{val:.4f}"
    return f"{val:.3f}"

rows = []
for _, r in summary.iterrows():
    config    = r["config"]
    proposed  = (config == PROPOSED)
    p_auc_str = "\u2014" if proposed else fmt_pval(r.get("p_auc_vs_proposed", np.nan))
    p_ap_str  = "\u2014" if proposed else fmt_pval(r.get("p_ap_vs_proposed",  np.nan))
    rows.append({
        "Configuration":  LABEL_MAP.get(config, config),
        "AUROC":          fmt(r["auc_roc_mean"], r["auc_roc_std"]),
        "AP":             fmt(r["ap_mean"],       r["ap_std"]),
        "Wins (AUROC)":   f"{int(r['wins_auc_roc'])} / {int(r['n_datasets'])}",
        "Wins (AP)":      f"{int(r['wins_ap'])} / {int(r['n_datasets'])}",
        "p (AUROC)":      p_auc_str,
        "p (AP)":         p_ap_str,
    })

table9 = pd.DataFrame(rows)
print("Table 9. ADBench Replication \u2014 Aggregate Results across 28 Datasets")
print("(mean \u00b1 std across 3 seeds per dataset, aggregated across datasets)\n")
print(table9.to_string(index=False))


Table 9. ADBench Replication — Aggregate Results across 28 Datasets
(mean ± std across 3 seeds per dataset, aggregated across datasets)

       Configuration         AUROC            AP Wins (AUROC) Wins (AP) p (AUROC) p (AP)
C_latent_diff_uncond 0.753 ± 0.174 0.346 ± 0.283       7 / 27    7 / 27     0.021  0.115
           E_caldm_d 0.777 ± 0.163 0.360 ± 0.289      12 / 27   11 / 27         —      —
           F_caldm_j 0.735 ± 0.157 0.279 ± 0.202       8 / 27    9 / 27     0.049  0.040


In [4]:
e_row = summary[summary["config"] == "E_caldm_d"].iloc[0]
c_row = summary[summary["config"] == "C_latent_diff_uncond"].iloc[0]
f_row = summary[summary["config"] == "F_caldm_j"].iloc[0]

print("=" * 62)
print("Section 7.5 — Aggregate result claims")
print("=" * 62)

auc_diff_ce = abs(e_row["auc_roc_mean"] - c_row["auc_roc_mean"])
print(f"\nAUROC diff E vs C : {auc_diff_ce:.4f}   paper: < 0.001")
print(f"p(AUROC) C vs E   : {c_row.get('p_auc_vs_proposed', np.nan):.3f}   paper: 0.940")
print(f"p(AP) C vs E      : {c_row.get('p_ap_vs_proposed',  np.nan):.3f}   paper: 0.381")

print(f"\nWins AUROC  C={int(c_row['wins_auc_roc'])}, E={int(e_row['wins_auc_roc'])}, "
      f"F={int(f_row['wins_auc_roc'])}   paper: 9-9-10")
print(f"Wins AP     C={int(c_row['wins_ap'])}, E={int(e_row['wins_ap'])}, "
      f"F={int(f_row['wins_ap'])}   paper: 8-10-10")

print(f"\nF AUROC: {f_row['auc_roc_mean']:.3f} \u00b1 {f_row['auc_roc_std']:.3f}   paper: 0.740 \u00b1 0.160")
print(f"p(AUROC) F vs E: {f_row.get('p_auc_vs_proposed', np.nan):.3f}   paper: 0.517")
print(f"p(AP)    F vs E: {f_row.get('p_ap_vs_proposed',  np.nan):.3f}   paper: 0.078")

print("\n" + "=" * 62)
print("Section 7.5 — Stability: variance does not replicate on ADBench")
print("=" * 62)

e_std = e_row["auc_roc_std"]
f_std = f_row["auc_roc_std"]
print(f"\nConfig F AUROC std (cross-dataset): {f_std:.3f}   paper: 0.160")
print(f"Config E AUROC std (cross-dataset): {e_std:.3f}   paper: 0.180")
print(f"F std < E std: {f_std < e_std}   paper: True (stability gap does not replicate)")

print("\n" + "=" * 62)
print("Section 7.5 — Per-dataset variation E vs F")
print("=" * 62)

e_ds = (per_run[per_run["config"] == "E_caldm_d"]
        .groupby("dataset")["auc_roc"].mean().reset_index()
        .rename(columns={"auc_roc": "E"}))
f_ds = (per_run[per_run["config"] == "F_caldm_j"]
        .groupby("dataset")["auc_roc"].mean().reset_index()
        .rename(columns={"auc_roc": "F"}))
ef   = e_ds.merge(f_ds, on="dataset")
ef["abs_diff"] = (ef["E"] - ef["F"]).abs()

n_gt05  = (ef["abs_diff"] > 0.05).sum()
n_gt020 = (ef["abs_diff"] > 0.20).sum()
print(f"\nDatasets where |E - F| > 0.05 : {n_gt05} / {len(ef)}   paper: 16 / 28")
print(f"Datasets where |E - F| > 0.20 : {n_gt020} / {len(ef)}   paper: 6 / 28")

Section 7.5 — Aggregate result claims

AUROC diff E vs C : 0.0246   paper: < 0.001
p(AUROC) C vs E   : 0.021   paper: 0.940
p(AP) C vs E      : 0.115   paper: 0.381

Wins AUROC  C=7, E=12, F=8   paper: 9-9-10
Wins AP     C=7, E=11, F=9   paper: 8-10-10

F AUROC: 0.735 ± 0.157   paper: 0.740 ± 0.160
p(AUROC) F vs E: 0.049   paper: 0.517
p(AP)    F vs E: 0.040   paper: 0.078

Section 7.5 — Stability: variance does not replicate on ADBench

Config F AUROC std (cross-dataset): 0.157   paper: 0.160
Config E AUROC std (cross-dataset): 0.163   paper: 0.180
F std < E std: True   paper: True (stability gap does not replicate)

Section 7.5 — Per-dataset variation E vs F

Datasets where |E - F| > 0.05 : 17 / 27   paper: 16 / 28
Datasets where |E - F| > 0.20 : 2 / 27   paper: 6 / 28
